In [0]:
%run ../config/config

In [0]:
dbutils.widgets.text("batch_id","")
batch_id=dbutils.widgets.get("batch_id")

Config file for catalog, schema, table names and landing zone path


In [0]:
bronze_table_complete_data = f"{catalog}.{bronze_schema}.complete_data"
bronze_table_active_weather = f"{catalog}.{bronze_schema}.active_weather"
bronze_table_cancellation = f"{catalog}.{bronze_schema}.cancellation"
bronze_table_stations= f"{catalog}.{bronze_schema}.stations"
bronze_table_carriers= f"{catalog}.{bronze_schema}.carriers"

Load quality check class

In [0]:
%run ../src/quality_check

Quality check class to inspect schema, missing - and duplicated values

Read bronze Delta table into a DataFrame

In [0]:
bronze_complete_df=(
    spark.read
    .format("delta")
    .table(bronze_table_complete_data)
)

Initiate SparkQCheck class and run get_schema() method to check schema definition

In [0]:
qc_bronze_complete=SparkQCheck(bronze_complete_df)
qc_bronze_complete.get_schema()

root
 |-- fl_date: date (nullable = true)
 |-- dep_hour: integer (nullable = true)
 |-- mkt_unique_carrier: string (nullable = true)
 |-- mkt_carrier_fl_num: integer (nullable = true)
 |-- op_unique_carrier: string (nullable = true)
 |-- op_carrier_fl_num: integer (nullable = true)
 |-- tail_num: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- dep_time: timestamp (nullable = true)
 |-- crs_dep_time: timestamp (nullable = true)
 |-- taxi_out: integer (nullable = true)
 |-- dep_delay: integer (nullable = true)
 |-- air_time: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- cancelled: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- elevation: integer (nullable = true)
 |-- mesonet_station: string (nullable = true)
 |-- year_of_manufacture: integer (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- icao_type: string (nullable = true)

Use the null_check_report() method to check missing data

According to the report, there are several columns with missing data
We will drop the columns, where 100% of the data is missing.
```
bash
|           wind_dir|   1065572|   100.0|
|           wind_spd|   1065572|   100.0|
|          wind_gust|   1065572|   100.0|
|         visibility|   1065572|   100.0|
|        temperature|      5960|    0.56|
|          dew_point|      5960|    0.56|
|       rel_humidity|      5960|    0.56|
|          altimeter|      5960|    0.56|
| lowest_cloud_layer|   1065572|   100.0|
|      n_cloud_layer|   1065572|   100.0|
|    low_level_cloud|   1065572|   100.0|
|    mid_level_cloud|   1065572|   100.0|
|   high_level_cloud|   1065572|   100.0|
|        cloud_cover|   1065572|   100.0|
|     active_weather|   1065572|   100.0|
```

In [0]:
qc_bronze_complete.null_check_report().show(50)

+-------------------+----------+--------+
|             column|null_count|null_pct|
+-------------------+----------+--------+
|            fl_date|         0|     0.0|
|           dep_hour|         0|     0.0|
| mkt_unique_carrier|         0|     0.0|
| mkt_carrier_fl_num|         0|     0.0|
|  op_unique_carrier|         0|     0.0|
|  op_carrier_fl_num|         0|     0.0|
|           tail_num|         0|     0.0|
|             origin|         0|     0.0|
|               dest|         0|     0.0|
|           dep_time|         0|     0.0|
|       crs_dep_time|         0|     0.0|
|           taxi_out|         0|     0.0|
|          dep_delay|         0|     0.0|
|           air_time|         0|     0.0|
|           distance|         0|     0.0|
|          cancelled|         0|     0.0|
|           latitude|         0|     0.0|
|          longitude|         0|     0.0|
|          elevation|         0|     0.0|
|    mesonet_station|         0|     0.0|
|year_of_manufacture|         0|  

We are investaging further the range column, where there are only 9 missing data

We can see that these are cancelled flights

Therefore, we will preserve these rows

In [0]:
bronze_complete_df.filter(F.col("range").isNull()).show()

+----------+--------+------------------+------------------+-----------------+-----------------+--------+------+----+-------------------+-------------------+--------+---------+--------+--------+---------+---------+----------+---------+---------------+-------------------+------------+---------+-----+-----------+--------+--------+---------+----------+-----------+---------+------------+---------+------------------+-------------+---------------+---------------+----------------+-----------+--------------+--------------------+--------------------+--------+
|   fl_date|dep_hour|mkt_unique_carrier|mkt_carrier_fl_num|op_unique_carrier|op_carrier_fl_num|tail_num|origin|dest|           dep_time|       crs_dep_time|taxi_out|dep_delay|air_time|distance|cancelled| latitude| longitude|elevation|mesonet_station|year_of_manufacture|manufacturer|icao_type|range|      width|wind_dir|wind_spd|wind_gust|visibility|temperature|dew_point|rel_humidity|altimeter|lowest_cloud_layer|n_cloud_layer|low_level_cloud|

Duplicate data check show no duplicated rows

In [0]:
qc_bronze_complete.duplicate_check_report().show(50)

+-------+--------+------------------+------------------+-----------------+-----------------+--------+------+----+--------+------------+--------+---------+--------+--------+---------+--------+---------+---------+---------------+-------------------+------------+---------+-----+-----+--------+--------+---------+----------+-----------+---------+------------+---------+------------------+-------------+---------------+---------------+----------------+-----------+--------------+-------------------+-----------+--------+-----+
|fl_date|dep_hour|mkt_unique_carrier|mkt_carrier_fl_num|op_unique_carrier|op_carrier_fl_num|tail_num|origin|dest|dep_time|crs_dep_time|taxi_out|dep_delay|air_time|distance|cancelled|latitude|longitude|elevation|mesonet_station|year_of_manufacture|manufacturer|icao_type|range|width|wind_dir|wind_spd|wind_gust|visibility|temperature|dew_point|rel_humidity|altimeter|lowest_cloud_layer|n_cloud_layer|low_level_cloud|mid_level_cloud|high_level_cloud|cloud_cover|active_weather|in

Read bronze Delta table into a DataFrame

In [0]:
bronze_active_weather_df=(
    spark.read
    .format("delta")
    .table(bronze_table_active_weather)
    )

Initiate SparkQCheck class and run get_schema() method to check schema definition

In [0]:
qc_bronze_active_weather=SparkQCheck(bronze_active_weather_df)
qc_bronze_active_weather.get_schema()

root
 |-- status: integer (nullable = true)
 |-- weather_description: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)



In [0]:
qc_bronze_active_weather.null_check_report().show(50)

+-------------------+----------+--------+
|             column|null_count|null_pct|
+-------------------+----------+--------+
|             status|         0|     0.0|
|weather_description|         0|     0.0|
|ingestion_timestamp|         0|     0.0|
|        source_file|         0|     0.0|
|           batch_id|         0|     0.0|
+-------------------+----------+--------+



Duplicate data check show no duplicated rows

In [0]:
qc_bronze_active_weather.duplicate_check_report().show(50)

+------+-------------------+-------------------+-----------+--------+-----+
|status|weather_description|ingestion_timestamp|source_file|batch_id|count|
+------+-------------------+-------------------+-----------+--------+-----+
+------+-------------------+-------------------+-----------+--------+-----+



Read bronze Delta table into a DataFrame

In [0]:
bronze_stations_df=(
    spark.read
    .format("delta")
    .table(bronze_table_stations)
)

Initiate SparkQCheck class and run get_schema() method to check schema definition

In [0]:
qc_bronze_stations=SparkQCheck(bronze_stations_df)
qc_bronze_stations.get_schema()

root
 |-- airport_id: integer (nullable = true)
 |-- airport: string (nullable = true)
 |-- display_airport_name: string (nullable = true)
 |-- display_airport_city_name_full: string (nullable = true)
 |-- airport_state_name: string (nullable = true)
 |-- airport_state_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- elevation: integer (nullable = true)
 |-- icao: string (nullable = true)
 |-- iata: string (nullable = true)
 |-- faa: string (nullable = true)
 |-- mesonet_station: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)



In [0]:
qc_bronze_stations.null_check_report().show()

+--------------------+----------+--------+
|              column|null_count|null_pct|
+--------------------+----------+--------+
|          airport_id|         0|     0.0|
|             airport|         0|     0.0|
|display_airport_name|         0|     0.0|
|display_airport_c...|         0|     0.0|
|  airport_state_name|         0|     0.0|
|  airport_state_code|         0|     0.0|
|            latitude|         0|     0.0|
|           longitude|         0|     0.0|
|           elevation|         0|     0.0|
|                icao|         0|     0.0|
|                iata|         0|     0.0|
|                 faa|         0|     0.0|
|     mesonet_station|         0|     0.0|
| ingestion_timestamp|         0|     0.0|
|         source_file|         0|     0.0|
|            batch_id|         0|     0.0|
+--------------------+----------+--------+



Duplicate data check show no duplicated rows

In [0]:
qc_bronze_stations.duplicate_check_report().show()

+----------+-------+--------------------+------------------------------+------------------+------------------+--------+---------+---------+----+----+---+---------------+-------------------+-----------+--------+-----+
|airport_id|airport|display_airport_name|display_airport_city_name_full|airport_state_name|airport_state_code|latitude|longitude|elevation|icao|iata|faa|mesonet_station|ingestion_timestamp|source_file|batch_id|count|
+----------+-------+--------------------+------------------------------+------------------+------------------+--------+---------+---------+----+----+---+---------------+-------------------+-----------+--------+-----+
+----------+-------+--------------------+------------------------------+------------------+------------------+--------+---------+---------+----+----+---+---------------+-------------------+-----------+--------+-----+



Read bronze Delta table into a DataFrame

In [0]:
bronze_carriers_df=(
    spark.read
    .format("delta")
    .table(bronze_table_carriers)
)

Initiate SparkQCheck class and run get_schema() method to check schema definition

In [0]:
qc_bronze_carriers=SparkQCheck(bronze_carriers_df)
qc_bronze_carriers.get_schema()

root
 |-- code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)



In [0]:
qc_bronze_carriers.null_check_report().show()

+-------------------+----------+--------+
|             column|null_count|null_pct|
+-------------------+----------+--------+
|               code|         0|     0.0|
|        description|         0|     0.0|
|ingestion_timestamp|         0|     0.0|
|        source_file|         0|     0.0|
|           batch_id|         0|     0.0|
+-------------------+----------+--------+



Duplicate data check show no duplicated rows

In [0]:
qc_bronze_carriers.duplicate_check_report().show()

+----+-----------+-------------------+-----------+--------+-----+
|code|description|ingestion_timestamp|source_file|batch_id|count|
+----+-----------+-------------------+-----------+--------+-----+
+----+-----------+-------------------+-----------+--------+-----+

